<a href="https://colab.research.google.com/github/tasninkhanlamha/SkillMorph/blob/main/stack%200.9237----.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import gc

from sklearn.model_selection import train_test_split
from sklearn.ensemble import StackingClassifier, RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

import pandas as pd
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# 1. Load dataset
df = pd.read_csv("/content/drive/MyDrive/Skill-Morph/android_ransomware_preprocessed.csv")

# 2. Cleaning
df = df.replace([np.inf, -np.inf], np.nan)
df.dropna(inplace=True)

# Duplicate remove
df = df.drop_duplicates()

# 3. Label binary
df['Label'] = df['Label'].apply(lambda x: 0 if x == 0 else 1)

# 4. Encode object columns
for col in df.select_dtypes(include=['object']).columns:
    df[col] = LabelEncoder().fit_transform(df[col].astype(str))

X = df.drop(columns=['Label'])
y = df['Label']

# 5. Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

# 6. Correct imbalance ratio
neg = sum(y_train == 0)
pos = sum(y_train == 1)
scale_weight = neg / pos

print("Negative:", neg)
print("Positive:", pos)
print("Correct XGB scale_pos_weight:", scale_weight)

# 7. Feature selection using LightGBM
selector = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    class_weight='balanced',
    random_state=42,
    n_jobs=1
)

selector.fit(X_train, y_train)

importance_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': selector.feature_importances_
})

# Top 50 features, not only 20
top_features = importance_df.sort_values(
    by='Importance',
    ascending=False
).head(50)['Feature'].tolist()

X_train_sel = X_train[top_features]
X_test_sel = X_test[top_features]

# 8. Corrected stacking model
estimators = [
    ('rf', RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        class_weight='balanced',
        random_state=42,
        n_jobs=1
    )),

    ('xgb', XGBClassifier(
        n_estimators=500,
        max_depth=8,
        learning_rate=0.03,
        subsample=0.9,
        colsample_bytree=0.9,
        scale_pos_weight=scale_weight,
        eval_metric='logloss',
        random_state=42,
        n_jobs=1
    )),

    ('et', ExtraTreesClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        class_weight='balanced',
        random_state=42,
        n_jobs=1
    ))
]

meta_clf = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=64,
    class_weight='balanced',
    random_state=42,
    n_jobs=1
)

stacking_clf = StackingClassifier(
    estimators=estimators,
    final_estimator=meta_clf,
    cv=5,
    stack_method='predict_proba',
    passthrough=True,
    n_jobs=1
)

# 9. Train
print("Training corrected stacking model...")
stacking_clf.fit(X_train_sel, y_train)

# 10. Threshold tuning
y_prob = stacking_clf.predict_proba(X_test_sel)[:, 1]

best_acc = 0
best_th = 0.5

for th in np.arange(0.05, 0.95, 0.01):
    y_pred = (y_prob >= th).astype(int)
    acc = accuracy_score(y_test, y_pred)

    if acc > best_acc:
        best_acc = acc
        best_th = th

print("\nBest Threshold:", best_th)
print("Best Accuracy:", best_acc)

# Final prediction
y_pred_final = (y_prob >= best_th).astype(int)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_final, digits=5))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_final))

Mounted at /content/drive


/tmp/ipykernel_5658/2735557437.py:29: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Label'] = df['Label'].apply(lambda x: 0 if x == 0 else 1)


Negative: 29774
Positive: 240465
Correct XGB scale_pos_weight: 0.12381843511529744
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 240465, number of negative: 29774
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.250216 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 14168
[LightGBM] [Info] Number of data points in the train set: 270239, number of used features: 66
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
Training corrected stacking model...
[LightGBM] [Info] Number of positive: 240465, number of negative: 29774
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.078620 seconds.
You can set `force_row_wise=true` to remove the overhead.
And i

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



Best Threshold: 0.13
Best Accuracy: 0.923785594639866

Classification Report:
              precision    recall  f1-score   support

           0    0.82901   0.38832   0.52890     12760
           1    0.92894   0.99008   0.95854    103058

    accuracy                        0.92379    115818
   macro avg    0.87898   0.68920   0.74372    115818
weighted avg    0.91793   0.92379   0.91120    115818


Confusion Matrix:
[[  4955   7805]
 [  1022 102036]]
